# Export current NLP report assets to existing Overleaf folder

Run the cells after you have generated the recovered report assets.

This notebook now matches the updated `report.tex` that uses the recovered/current files only. It no longer expects the old deleted figures such as `class_distribution.png`, `clause_length_histogram.png`, `model_comparison_accuracy.png`, `validation_vs_test_macro_f1.png`, `per_label_f1_best_model.png`, or `llm_model_comparison.png`.


In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
from pathlib import Path

REPO = Path("/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing")

expected = [
    "report.tex",
    "figures/label_distribution.png",
    "figures/clause_length_distribution.png",
    "figures/model_comparison_macro_f1.png",
    "figures/model_comparison_macro_f1_with_incomplete.png",
    "figures/hpt_best_validation_macro_f1.png",
    "figures/qwen_invalid_predictions.png",
    "figures/confusion_matrix_best_model.png",
    "outputs/main_results.csv",
    "outputs/misclassified_examples.csv",
    "outputs/confusion_pairs.csv",
]

print("Repo exists:", REPO.exists())
print("Repo path:", REPO)

for f in expected:
    print(("✅ " if (REPO / f).exists() else "❌ ") + f)

Repo exists: True
Repo path: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing
✅ report.tex
✅ figures/label_distribution.png
✅ figures/clause_length_distribution.png
✅ figures/model_comparison_macro_f1.png
✅ figures/model_comparison_macro_f1_with_incomplete.png
✅ figures/hpt_best_validation_macro_f1.png
✅ figures/qwen_invalid_predictions.png
✅ figures/confusion_matrix_best_model.png
✅ outputs/main_results.csv
✅ outputs/misclassified_examples.csv
✅ outputs/confusion_pairs.csv


In [3]:
from pathlib import Path
import shutil
import re
import pandas as pd



REPO = Path("/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing")
OVERLEAF = REPO / "overleaf_export"

# Set to False if you only want to export assets and keep the current Overleaf report.tex untouched.
COPY_REPORT_TEX = True

# Keep this False so the notebook prints missing files instead of crashing.
# The updated report.tex uses \safeincludegraphics, so missing images become visible placeholders in Overleaf.
STRICT = False

OVERLEAF.mkdir(parents=True, exist_ok=True)

# Core Overleaf/ACL files.
CORE_FILES = {
    "report.tex",
    "custom.bib",
    "anthology.bib",
    "EACL2023.sty",
    "acl_natbib.bst",
}

# Current regenerated figures used by the updated report.tex.
CURRENT_FIGURES = {
    "figures/label_distribution.png",
    "figures/clause_length_distribution.png",
    "figures/model_comparison_macro_f1.png",
    "figures/model_comparison_macro_f1_with_incomplete.png",
    "figures/hpt_best_validation_macro_f1.png",
    "figures/qwen_invalid_predictions.png",
    "figures/confusion_matrix_best_model.png",
}

# Current CSV/report evidence produced by the recovery/error-analysis notebook.
CURRENT_OUTPUTS = {
    "outputs/main_results.csv",
    "outputs/recovered_final_model_comparison.csv",
    "outputs/recovered_prediction_metrics.csv",
    "outputs/per_class_results.csv",
    "outputs/misclassified_examples.csv",
    "outputs/confusion_pairs.csv",
    "outputs/recovered_misclassified_examples.csv",
    "outputs/recovered_top_confusions.csv",
    "outputs/recovery_audit.csv",
    "outputs/class_imbalance.csv",
    "outputs/hpt_best_trials_report_table.csv",
    "outputs/figure_reference_audit.csv",
}

# Optional supporting outputs that the report may mention as evidence, but does not need for compilation.
OPTIONAL_OUTPUTS = {
    "outputs/eda/top_tfidf_terms_per_label.csv",
    "outputs/hpt_summary.csv",
    "outputs/classical_hyperparameter_results.csv",
}

KEEP = CORE_FILES | CURRENT_FIGURES | CURRENT_OUTPUTS | OPTIONAL_OUTPUTS

# Old/deleted assets that should no longer be carried into the Overleaf export.
STALE_ASSETS = {
    "figures/class_distribution.png",
    "figures/clause_length_histogram.png",
    "figures/model_comparison_accuracy.png",
    "figures/hpt_validation_macro_f1.png",
    "figures/per_label_f1_best_model.png",
    "figures/validation_vs_test_macro_f1.png",
    "figures/llm_model_comparison.png",
}

In [4]:
def source_candidates(rel: str) -> list[Path]:
    """
    Return possible source locations for a report asset.

    The report expects figures under figures/, but the recovery notebook also duplicates
    them under outputs/figures/ and sometimes outputs/. This fallback avoids silly path pain.
    """
    rel_path = Path(rel)
    candidates = [REPO / rel_path]

    if rel.startswith("figures/"):
        filename = rel_path.name
        candidates.extend([
            REPO / "outputs" / "figures" / filename,
            REPO / "outputs" / filename,
        ])

    return candidates


def copy_one(rel: str) -> tuple[str, str | None]:
    """
    Copy one file into overleaf_export preserving its relative path.

    Returns:
        ("copied", source_path) if copied
        ("kept_existing", existing_path) if already present and no source found
        ("missing", None) if neither source nor existing target exists
        ("skipped_report", source_path) if report.tex copy disabled
    """
    src = None
    for candidate in source_candidates(rel):
        if candidate.exists() and candidate.is_file():
            src = candidate
            break

    dst = OVERLEAF / rel

    if rel == "report.tex" and not COPY_REPORT_TEX:
        return ("skipped_report", str(src) if src else None)

    if src is not None:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        return ("copied", str(src))

    if dst.exists() and dst.is_file():
        return ("kept_existing", str(dst))

    return ("missing", None)


copied, kept_existing, skipped, missing = [], [], [], []

for rel in sorted(KEEP):
    status, src = copy_one(rel)
    if status == "copied":
        copied.append((rel, src))
    elif status == "kept_existing":
        kept_existing.append(rel)
    elif status == "skipped_report":
        skipped.append((rel, src))
    else:
        missing.append(rel)

# Delete stale/unexpected files from overleaf_export only.
# This keeps the export compact and prevents old duplicate figures from being accidentally submitted.
deleted = []
for file in OVERLEAF.rglob("*"):
    if not file.is_file():
        continue
    rel = file.relative_to(OVERLEAF).as_posix()
    if rel not in KEEP:
        file.unlink()
        deleted.append(rel)

# Remove empty folders.
for folder in sorted([p for p in OVERLEAF.rglob("*") if p.is_dir()], reverse=True):
    try:
        folder.rmdir()
    except OSError:
        pass

print("Export complete.")
print(f"Copied files: {len(copied)}")
print(f"Kept existing files: {len(kept_existing)}")
print(f"Skipped files: {len(skipped)}")
print(f"Deleted stale/unexpected files: {len(deleted)}")
print(f"Missing expected/optional files: {len(missing)}")

if copied:
    print("\nCopied:")
    for rel, src in copied:
        print(f"✅ {rel}  <-  {src}")

if kept_existing:
    print("\nKept existing because no source copy was found:")
    for rel in kept_existing:
        print(f"⚠️ {rel}")

if deleted:
    print("\nDeleted from overleaf_export:")
    for rel in deleted:
        print(f"🗑️ {rel}")

if missing:
    print("\nMissing:")
    for rel in missing:
        # Current required figures are more important than optional CSV evidence.
        marker = "❌ REQUIRED" if rel in CURRENT_FIGURES or rel in CORE_FILES else "⚠️ optional/evidence"
        print(f"{marker}: {rel}")

if STRICT and missing:
    raise FileNotFoundError("Some expected report assets are missing. See list above.")

Export complete.
Copied files: 22
Kept existing files: 5
Skipped files: 0
Deleted stale/unexpected files: 37
Missing expected/optional files: 0

Copied:
✅ figures/clause_length_distribution.png  <-  /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/figures/clause_length_distribution.png
✅ figures/confusion_matrix_best_model.png  <-  /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/figures/confusion_matrix_best_model.png
✅ figures/hpt_best_validation_macro_f1.png  <-  /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/figures/hpt_best_validation_macro_f1.png
✅ figures/label_distribution.png  <-  /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/figures/label_distribution.png
✅ figures/model_comparison_macro_f1.png  <-  /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/figures/model_comparison_macro_f1.png
✅ figures/model_comparison_ma

In [5]:
# Audit report.tex against exported figures.

tex_path = OVERLEAF / "report.tex"
old_names = sorted(STALE_ASSETS)

if tex_path.exists():
    tex = tex_path.read_text(encoding="utf-8", errors="replace")

    # Works for \includegraphics{...} and \safeincludegraphics[...]{...}
    figure_refs = sorted(set(re.findall(r"\\(?:safeincludegraphics|includegraphics)(?:\[[^\]]*\])?\{([^}]+)\}", tex)))
    # Exclude macro definition placeholder {#2}
    figure_refs = [f for f in figure_refs if not f.startswith("#")]

    missing_fig_refs = [f for f in figure_refs if not (OVERLEAF / f).exists()]

    stale_refs = [f for f in old_names if f in tex]

    print("Report figure references:")
    for f in figure_refs:
        print(("✅ " if (OVERLEAF / f).exists() else "❌ ") + f)

    if stale_refs:
        print("\nOld/deleted figure names still referenced in report.tex:")
        for f in stale_refs:
            print(f"❌ {f}")
    else:
        print("\n✅ No old/deleted figure names are referenced in report.tex.")

    if missing_fig_refs:
        print("\nMissing figure files referenced by report.tex:")
        for f in missing_fig_refs:
            print(f"❌ {f}")
    else:
        print("\n✅ All figure files referenced by report.tex exist in overleaf_export.")

else:
    print("❌ overleaf_export/report.tex was not found.")

Report figure references:
❌ figures/class_distribution.png
❌ figures/clause_length_histogram.png
✅ figures/confusion_matrix_best_model.png
✅ figures/hpt_best_validation_macro_f1.png
❌ figures/hpt_validation_macro_f1.png
❌ figures/label_similarity_heatmap.png
❌ figures/llm_model_comparison.png
❌ figures/model_comparison_accuracy.png
✅ figures/model_comparison_macro_f1.png
✅ figures/model_comparison_macro_f1_with_incomplete.png
❌ figures/per_label_f1_best_model.png
✅ figures/qwen_invalid_predictions.png
❌ figures/validation_vs_test_macro_f1.png

Old/deleted figure names still referenced in report.tex:
❌ figures/class_distribution.png
❌ figures/clause_length_histogram.png
❌ figures/hpt_validation_macro_f1.png
❌ figures/llm_model_comparison.png
❌ figures/model_comparison_accuracy.png
❌ figures/per_label_f1_best_model.png
❌ figures/validation_vs_test_macro_f1.png

Missing figure files referenced by report.tex:
❌ figures/class_distribution.png
❌ figures/clause_length_histogram.png
❌ figures/

In [6]:
# Create a fresh ZIP for upload to Overleaf.

zip_path = Path(shutil.make_archive(str(OVERLEAF), "zip", OVERLEAF))

print("ZIP created:")
print(zip_path)

# Show compact tree of exported package.
print("\nExported package tree:")
for path in sorted(OVERLEAF.rglob("*")):
    rel = path.relative_to(OVERLEAF)
    if path.is_file():
        size_kb = path.stat().st_size / 1024
        print(f"- {rel.as_posix()} ({size_kb:.1f} KB)")

ZIP created:
/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/overleaf_export.zip

Exported package tree:
- EACL2023.sty (11.0 KB)
- acl_natbib.bst (46.6 KB)
- anthology.bib (41128.2 KB)
- custom.bib (0.9 KB)
- figures/clause_length_distribution.png (74.8 KB)
- figures/confusion_matrix_best_model.png (252.3 KB)
- figures/hpt_best_validation_macro_f1.png (81.9 KB)
- figures/label_distribution.png (183.4 KB)
- figures/model_comparison_macro_f1.png (227.6 KB)
- figures/model_comparison_macro_f1_with_incomplete.png (229.3 KB)
- figures/qwen_invalid_predictions.png (78.0 KB)
- outputs/class_imbalance.csv (0.3 KB)
- outputs/classical_hyperparameter_results.csv (1.1 KB)
- outputs/confusion_pairs.csv (25.8 KB)
- outputs/eda/top_tfidf_terms_per_label.csv (15.7 KB)
- outputs/figure_reference_audit.csv (1.3 KB)
- outputs/hpt_best_trials_report_table.csv (0.4 KB)
- outputs/hpt_summary.csv (0.3 KB)
- outputs/main_results.csv (8.1 KB)
- outputs/misclassified_examples.

In [7]:
from pathlib import Path

REPO = Path("/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing")

print("REPO exists:", REPO.exists())
print("REPO:", REPO)

print("\nRepo root:")
if REPO.exists():
    for p in REPO.iterdir():
        print(" -", p.name, "DIR" if p.is_dir() else "FILE")

print("\nSearching key files:")
targets = [
    "report.tex",
    "label_distribution.png",
    "clause_length_distribution.png",
    "model_comparison_macro_f1.png",
    "confusion_matrix_best_model.png",
    "main_results.csv",
    "misclassified_examples.csv",
    "confusion_pairs.csv",
]

for target in targets:
    matches = list(Path("/content/drive/MyDrive").rglob(target))
    print(f"\n{target}: {len(matches)}")
    for m in matches[:10]:
        print(" ", m)

REPO exists: True
REPO: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing

Repo root:
 - .gitattributes FILE
 - AGENTS.md FILE
 - README.md FILE
 - README_COLAB.md FILE
 - checkpoints DIR
 - data DIR
 - download ledgar.py FILE
 - figures DIR
 - inference DIR
 - models DIR
 - modules DIR
 - notebooks DIR
 - outputs DIR
 - pyproject.toml FILE
 - requirements-colab.txt FILE
 - requirements.txt FILE
 - results DIR
 - scripts DIR
 - src DIR
 - tests DIR
 - wandb DIR
 - .gitignore FILE
 - overleaf_export DIR
 - report.tex FILE
 - export.ipynb FILE
 - overleaf_export.zip FILE

Searching key files:

report.tex: 2
  /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/report.tex
  /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/overleaf_export/report.tex

label_distribution.png: 9
  /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/figures/label_distribution.png
  /content/